In [1]:
import matplotlib as mpl
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
import sklearn
import pandas as pd
import os
import sys
import time
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from collections import Counter
from torch.utils.tensorboard import SummaryWriter
import torch.multiprocessing as mp

# 基础配置
device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
print(f"Using device: {device}")

seed = 42
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)

# CPU 线程与多进程配置（Windows/Jupyter 下避免死锁）
torch.set_num_threads(8)
torch.set_num_interop_threads(8)
if os.name == "nt":
    try:
        mp.set_start_method("spawn", force=True)
    except RuntimeError:
        pass
    mp.set_sharing_strategy("file_system")


Using device: cuda:0


In [2]:
# 1. 数据预处理函数
import unicodedata
import re

def unicode_to_ascii(s):
    return ''.join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn')

def preprocess_sentence(w):
    w = unicode_to_ascii(w.lower().strip())
    w = re.sub(r"([?.!,¿])", r" \1 ", w)
    w = re.sub(r'[" "]+', " ", w)
    w = re.sub(r"[^a-zA-Z?.!,¿]+", " ", w)
    w = w.rstrip().strip()
    return w


In [3]:
# 2. 数据集 definition
class LangPairDataset(Dataset):
    fpath = Path(r"./spa-eng/spa.txt")
    cache_path = Path(r"./.cache/lang_pair.npy")

    def _build_cache(self):
        self.cache_path.parent.mkdir(parents=True, exist_ok=True)
        with open(self.fpath, "r", encoding="utf8") as file:
            lines = file.readlines()
            lang_pair = [[preprocess_sentence(w) for w in l.split('\t')[:2]] for l in lines if l.strip()]
            trg, src = zip(*lang_pair)
            trg = np.array(trg)
            src = np.array(src)
            rng = np.random.default_rng(seed)
            split_index = rng.choice(a=["train", "test"], replace=True, p=[0.9, 0.1], size=len(src))
        np.save(self.cache_path, {"trg": trg, "src": src, "split_index": split_index})
        return trg, src, split_index

    def __init__(self, mode="train", cache=False):
        if cache or not self.cache_path.exists():
            trg, src, split_index = self._build_cache()
        else:
            lang_pair = np.load(self.cache_path, allow_pickle=True).item()
            trg = lang_pair.get("trg")
            src = lang_pair.get("src")
            split_index = lang_pair.get("split_index")
            invalid_cache = (
                trg is None
                or src is None
                or split_index is None
                or len(trg) != len(src)
                or len(split_index) != len(src)
            )
            if invalid_cache:
                trg, src, split_index = self._build_cache()
            else:
                split_index = np.asarray(split_index)

        self.split_index = split_index
        self.trg = trg[self.split_index == mode]
        self.src = src[self.split_index == mode]

    def __getitem__(self, index):
        return self.src[index], self.trg[index]

    def __len__(self):
        return len(self.src)

# 强制刷新缓存并加载数据
try:
    if Path(r"./.cache/lang_pair.npy").exists():
        os.remove(r"./.cache/lang_pair.npy")
except:
    pass

train_ds = LangPairDataset("train", cache=True)
test_ds = LangPairDataset("test")


In [4]:
# 3. 词表构建
def get_word_idx(ds, mode="src", threshold=1):
    word2idx = {"[PAD]": 0, "[BOS]": 1, "[UNK]": 2, "[EOS]": 3}
    idx2word = {0: "[PAD]", 1: "[BOS]", 2: "[UNK]", 3: "[EOS]"}
    index = 4
    word_list = " ".join([pair[0 if mode=="src" else 1] for pair in ds]).split()
    counter = Counter(word_list)
    for token, count in counter.items():
        if count >= threshold:
            word2idx[token] = index
            idx2word[index] = token
            index += 1
    return word2idx, idx2word

src_word2idx, src_idx2word = get_word_idx(train_ds, "src")
trg_word2idx, trg_idx2word = get_word_idx(train_ds, "trg")


In [5]:
# 4. Tokenizer
class Tokenizer:
    def __init__(self, word2idx, idx2word, max_length=500, pad_idx=0, bos_idx=1, eos_idx=3, unk_idx=2):
        self.word2idx = word2idx
        self.idx2word = idx2word
        self.max_length = max_length
        self.pad_idx = pad_idx
        self.bos_idx = bos_idx
        self.eos_idx = eos_idx
        self.unk_idx = unk_idx

    def encode(self, text_list, padding_first=False, add_bos=True, add_eos=True, return_mask=False):
        max_length = min(self.max_length, add_eos + add_bos + max([len(text) for text in text_list]))
        indices_list = []
        for text in text_list:
            indices = [self.word2idx.get(word, self.unk_idx) for word in text[:max_length - add_bos - add_eos]]
            if add_bos: indices = [self.bos_idx] + indices
            if add_eos: indices = indices + [self.eos_idx]
            padding = [self.pad_idx] * (max_length - len(indices))
            if padding_first: indices = padding + indices
            else: indices = indices + padding
            indices_list.append(indices)
        input_ids = torch.tensor(indices_list)
        masks = (input_ids == self.pad_idx).to(dtype=torch.int64)
        return input_ids if not return_mask else (input_ids, masks)

    def decode(self, indices_list, remove_bos=True, remove_eos=True, remove_pad=True, split=False):
        text_list = []
        for indices in indices_list:
            text = []
            for index in indices:
                word = self.idx2word.get(index, "[UNK]")
                if remove_bos and word == "[BOS]": continue
                if remove_eos and word == "[EOS]": break
                if remove_pad and word == "[PAD]": break
                text.append(word)
            text_list.append(" ".join(text) if not split else text)
        return text_list

src_tokenizer = Tokenizer(word2idx=src_word2idx, idx2word=src_idx2word)
trg_tokenizer = Tokenizer(word2idx=trg_word2idx, idx2word=trg_idx2word)

def collate_fct(batch):
    src_words = [pair[0].split() for pair in batch]
    trg_words = [pair[1].split() for pair in batch]
    encoder_inputs, encoder_inputs_mask = src_tokenizer.encode(src_words, padding_first=True, add_bos=True, add_eos=True, return_mask=True)
    decoder_inputs = trg_tokenizer.encode(trg_words, padding_first=False, add_bos=True, add_eos=False, return_mask=False)
    decoder_labels, decoder_labels_mask = trg_tokenizer.encode(trg_words, padding_first=False, add_bos=False, add_eos=True, return_mask=True)
    return {
        "encoder_inputs": encoder_inputs.to(device),
        "encoder_inputs_mask": encoder_inputs_mask.to(device),
        "decoder_inputs": decoder_inputs.to(device),
        "decoder_labels": decoder_labels.to(device),
        "decoder_labels_mask": decoder_labels_mask.to(device),
    }


In [6]:
# 5. 模型组件 (4层结构)

class Encoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim=256, hidden_dim=1024, num_layers=4): # 4层
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        # 增加 dropout 防止深层过拟合
        self.gru = nn.GRU(embedding_dim, hidden_dim, num_layers=num_layers, batch_first=True, dropout=0.5)

    def forward(self, encoder_inputs):
        embeds = self.embedding(encoder_inputs)
        seq_output, hidden = self.gru(embeds)
        return seq_output, hidden

class BahdanauAttention(nn.Module):
    def __init__(self, hidden_dim=1024):
        super().__init__()
        self.Wk = nn.Linear(hidden_dim, hidden_dim)
        self.Wq = nn.Linear(hidden_dim, hidden_dim)
        self.V = nn.Linear(hidden_dim, 1)

    def forward(self, query, keys, values, attn_mask=None):
        # 加入温度系数
        temperature = 1.2
        scores = self.V(F.tanh(self.Wk(keys) + self.Wq(query.unsqueeze(-2))))
        scores = scores / temperature

        if attn_mask is not None:
            attn_mask = (attn_mask.unsqueeze(-1)) * -1e16
            scores += attn_mask

        scores = F.softmax(scores, dim=-2)
        context_vector = torch.mul(scores, values).sum(dim=-2)
        return context_vector, scores

class Decoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim=256, hidden_dim=1024, num_layers=4): # 4层
        super(Decoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        # 增加 dropout
        self.gru = nn.GRU(embedding_dim + hidden_dim, hidden_dim, num_layers=num_layers, batch_first=True, dropout=0.5)
        self.fc = nn.Linear(hidden_dim, vocab_size)
        self.dropout = nn.Dropout(0.5)
        self.attention = BahdanauAttention(hidden_dim)

    def forward(self, decoder_input, hidden, encoder_outputs, attn_mask=None):
        context_vector, attention_score = self.attention(query=hidden, keys=encoder_outputs, values=encoder_outputs, attn_mask=attn_mask)
        embeds = self.embedding(decoder_input)
        embeds = torch.cat([context_vector.unsqueeze(-2), embeds], dim=-1)
        seq_output, hidden = self.gru(embeds)
        logits = self.fc(self.dropout(seq_output))
        return logits, hidden, attention_score

class Sequence2Sequence(nn.Module):
    def __init__(self, src_vocab_size, trg_vocab_size, encoder_embedding_dim=256, encoder_hidden_dim=1024, encoder_num_layers=4, decoder_embedding_dim=256, decoder_hidden_dim=1024, decoder_num_layers=4, bos_idx=1, eos_idx=3, max_length=512):
        super(Sequence2Sequence, self).__init__()
        self.bos_idx = bos_idx
        self.eos_idx = eos_idx
        self.max_length = max_length
        self.encoder = Encoder(src_vocab_size, embedding_dim=encoder_embedding_dim, hidden_dim=encoder_hidden_dim, num_layers=encoder_num_layers)
        self.decoder = Decoder(trg_vocab_size, embedding_dim=decoder_embedding_dim, hidden_dim=decoder_hidden_dim, num_layers=decoder_num_layers)

    def forward(self, *, encoder_inputs, decoder_inputs, attn_mask=None):
        encoder_outputs, hidden = self.encoder(encoder_inputs)
        bs, seq_len = decoder_inputs.shape
        logits_list = []
        scores_list = []
        for i in range(seq_len):
            logits, hidden, score = self.decoder(decoder_inputs[:, i:i+1], hidden[-1], encoder_outputs, attn_mask=attn_mask)
            logits_list.append(logits)
            scores_list.append(score)
        return torch.cat(logits_list, dim=-2), torch.cat(scores_list, dim=-1)

    @torch.no_grad()
    def infer(self, encoder_input, attn_mask=None):
        encoder_outputs, hidden = self.encoder(encoder_input)
        decoder_input = torch.Tensor([self.bos_idx]).reshape(1, 1).to(dtype=torch.int64).to(device)

        pred_list = []
        score_list = []

        for _ in range(self.max_length):
            logits, hidden, score = self.decoder(decoder_input, hidden[-1], encoder_outputs, attn_mask=attn_mask)

            decoder_pred = logits.argmax(dim=-1)
            decoder_input = decoder_pred

            token_id = decoder_pred.reshape(-1).item()
            pred_list.append(token_id)
            score_list.append(score)

            if token_id == self.eos_idx:
                break

        return pred_list, torch.cat(score_list, dim=-1)


In [7]:
# 6. 训练辅助
def cross_entropy_with_padding(logits, labels, padding_mask=None):
    bs, seq_len, nc = logits.shape
    loss = F.cross_entropy(logits.reshape(bs * seq_len, nc), labels.reshape(-1), reduce=False)
    if padding_mask is None: loss = loss.mean()
    else:
        padding_mask = 1 - padding_mask.reshape(-1)
        loss = torch.mul(loss, padding_mask).sum() / padding_mask.sum()
    return loss

class TensorBoardCallback:
    def __init__(self, log_dir, flush_secs=10):
        self.writer = SummaryWriter(log_dir=log_dir, flush_secs=flush_secs)
    def __call__(self, step, **kwargs):
        loss = kwargs.pop("loss", None)
        val_loss = kwargs.pop("val_loss", None)
        if loss is not None and val_loss is not None:
            self.writer.add_scalars("training/loss", {"loss": loss, "val_loss": val_loss}, step)
        lr = kwargs.pop("lr", None)
        if lr is not None: self.writer.add_scalars("training/lr", {"lr": lr}, step)

class SaveCheckpointsCallback:
    def __init__(self, save_dir, save_step=5000, save_best_only=True):
        self.save_dir = save_dir
        self.save_step = save_step
        self.save_best_only = save_best_only
        self.best_metrics = - np.inf
        if not os.path.exists(self.save_dir): os.makedirs(self.save_dir, exist_ok=True)
    def __call__(self, step, state_dict, metric=None):
        if step % self.save_step > 0: return
        if self.save_best_only:
            if metric >= self.best_metrics:
                torch.save(state_dict, os.path.join(self.save_dir, "best.ckpt"))
                self.best_metrics = metric
        else: torch.save(state_dict, os.path.join(self.save_dir, f"{step}.ckpt"))

class EarlyStopCallback:
    def __init__(self, patience=5, min_delta=0.01):
        self.patience = patience
        self.min_delta = min_delta
        self.best_metric = - np.inf
        self.counter = 0
    def __call__(self, metric):
        if metric >= self.best_metric + self.min_delta:
            self.best_metric = metric
            self.counter = 0
        else: self.counter += 1
    @property
    def early_stop(self): return self.counter >= self.patience

@torch.no_grad()
def evaluating(model, dataloader, loss_fct):
    loss_list = []
    for batch in dataloader:
        logits, _ = model(encoder_inputs=batch["encoder_inputs"], decoder_inputs=batch["decoder_inputs"], attn_mask=batch["encoder_inputs_mask"])
        loss = loss_fct(logits, batch["decoder_labels"], padding_mask=batch["decoder_labels_mask"])
        loss_list.append(loss.cpu().item())
    return np.mean(loss_list)

def training(model, train_loader, val_loader, epoch, loss_fct, optimizer, scheduler=None, tensorboard_callback=None, save_ckpt_callback=None, early_stop_callback=None, eval_step=500):
    record_dict = {"train": [], "val": []}
    global_step = 1
    model.train()
    with tqdm(total=epoch * len(train_loader)) as pbar:
        for epoch_id in range(epoch):
            for batch in train_loader:
                optimizer.zero_grad()
                logits, _ = model(encoder_inputs=batch["encoder_inputs"], decoder_inputs=batch["decoder_inputs"], attn_mask=batch["encoder_inputs_mask"])
                loss = loss_fct(logits, batch["decoder_labels"], padding_mask=batch["decoder_labels_mask"])
                loss.backward()

                # 【优化点 3】梯度裁剪
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

                optimizer.step()
                loss_val = loss.cpu().item()
                record_dict["train"].append({"loss": loss_val, "step": global_step})
                if global_step % eval_step == 0:
                    model.eval()
                    val_loss = evaluating(model, val_loader, loss_fct)
                    record_dict["val"].append({"loss": val_loss, "step": global_step})
                    model.train()
                    if scheduler: scheduler.step(val_loss)
                    if tensorboard_callback: tensorboard_callback(global_step, loss=loss_val, val_loss=val_loss, lr=optimizer.param_groups[0]["lr"])
                    if save_ckpt_callback: save_ckpt_callback(global_step, model.state_dict(), metric=-val_loss)
                    if early_stop_callback:
                        early_stop_callback(-val_loss)
                        if early_stop_callback.early_stop: return record_dict
                global_step += 1
                pbar.update(1)
            pbar.set_postfix({"epoch": epoch_id, "loss": loss_val})
    return record_dict

# 【优化点 4】正交初始化
def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None: nn.init.zeros_(m.bias)
    elif isinstance(m, nn.GRU):
        for name, param in m.named_parameters():
            if 'weight_hh' in name: nn.init.orthogonal_(param.data)
            elif 'weight_ih' in name: nn.init.xavier_uniform_(param.data)
            elif 'bias' in name: nn.init.zeros_(param.data)


In [ ]:
# 7. 开始训练
if __name__ == '__main__':
    epoch = 50
    batch_size = 64

    # 【优化点 1】4层结构
    model = Sequence2Sequence(
        src_vocab_size=len(src_word2idx),
        trg_vocab_size=len(trg_word2idx),
        encoder_num_layers=4,
        decoder_num_layers=4
    )
    model = model.to(device)
    model.apply(init_weights)

    # 多进程加载（8 workers）并保持 worker 常驻避免卡死
    train_dl = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collate_fct,
        num_workers=8,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=2,
    )
    test_dl = DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_fct,
        num_workers=8,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=2,
    )

    # 【优化点 2】AdamW + 0.0005 学习率
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.0005, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, verbose=True)

    exp_name = "seq2seq-4layers"
    tensorboard_callback = TensorBoardCallback(f"runs/{exp_name}")
    save_ckpt_callback = SaveCheckpointsCallback(f"checkpoints/{exp_name}", save_step=200, save_best_only=True)
    early_stop_callback = EarlyStopCallback(patience=8)

    print("开始训练 (4层, AdamW, lr=0.0005)...")
    record = training(model, train_dl, test_dl, epoch, cross_entropy_with_padding, optimizer, scheduler, tensorboard_callback, save_ckpt_callback, early_stop_callback)

    # 8. 绘制曲线
    plt.figure(figsize=(10, 5))
    plt.plot([i["step"] for i in record["train"]], [i["loss"] for i in record["train"]], label="train loss", alpha=0.5)
    plt.plot([i["step"] for i in record["val"]], [i["loss"] for i in record["val"]], label="val loss", linewidth=2)
    plt.legend(); plt.grid(); plt.show()

    # 9. 测试
    model.load_state_dict(torch.load(f"checkpoints/{exp_name}/best.ckpt", map_location=device))
    class Translator:
        def __init__(self, model, src_tokenizer, trg_tokenizer):
            self.model = model
            self.model.eval()
            self.src_tokenizer = src_tokenizer
            self.trg_tokenizer = trg_tokenizer
        def draw_attention_map(self, scores, src_words_list, trg_words_list):
            fig, ax = plt.subplots(figsize=(10, 10))
            cax = ax.matshow(scores.T, cmap='viridis')
            fig.colorbar(cax)
            ax.set_xticks(np.arange(len(src_words_list)))
            ax.set_yticks(np.arange(len(trg_words_list)))
            ax.set_xticklabels(src_words_list, rotation=90)
            ax.set_yticklabels(trg_words_list)
            for i in range(scores.shape[0]):
                for j in range(scores.shape[1]):
                    ax.text(j, i, f'{scores[i, j]:.2f}', ha='center', va='center', color='w' if scores[i, j] < 0.5 else 'k')
            plt.show()
        def __call__(self, sentence, show_attention=False):
            sentence = preprocess_sentence(sentence)
            encoder_input, attn_mask = self.src_tokenizer.encode([sentence.split()], padding_first=True, add_bos=True, add_eos=True, return_mask=True)
            encoder_input = torch.Tensor(encoder_input).to(dtype=torch.int64).to(device)
            attn_mask = attn_mask.to(device)
            preds, scores = self.model.infer(encoder_input=encoder_input, attn_mask=attn_mask)
            trg_sentence = self.trg_tokenizer.decode([preds], split=True, remove_eos=False)[0]
            if show_attention:
                src_decoded = self.src_tokenizer.decode(encoder_input.tolist(), split=True, remove_bos=False, remove_eos=False)[0]
                self.draw_attention_map(scores.squeeze(0).cpu().numpy(), src_decoded, trg_sentence)
            return " ".join(trg_sentence[:-1])

    translator = Translator(model, src_tokenizer, trg_tokenizer)
    print("-" * 30)
    print("【4层模型翻译结果】")
    test_sentences = [u'hace mucho frio aqui .', u'¿ puedo tomar prestado este libro ?', u'el gato esta sobre la mesa .', u'El hombre con sombrero es un médico.']
    for s in test_sentences:
        print(f"原文: {s}")
        show_attn = (s == test_sentences[0])
        print(f"翻译: {translator(s, show_attention=show_attn)}")
        print("-" * 30)

    # 10. BLEU
    try:
        import nltk
    except ImportError:
        import subprocess
        subprocess.check_call([sys.executable, "-m", "pip", "install", "nltk"])
        import nltk
    from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction, corpus_bleu
    def evaluate_bleu(model, test_ds, num_samples=100):
        print(f"开始评估 BLEU，抽取 {num_samples} 个样本...")
        indices = np.random.choice(len(test_ds), num_samples, replace=False)
        references = []
        candidates = []
        for idx in tqdm(indices):
            src_sent, trg_sent = test_ds[idx]
            ref = [trg_sent.split()]
            references.append(ref)
            pred_str = translator(src_sent, show_attention=False)
            cand = pred_str.split()
            candidates.append(cand)
        smooth = SmoothingFunction().method1
        score = corpus_bleu(references, candidates, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smooth)
        print(f"测试集 ({num_samples} 样本) 平均 BLEU 分数: {score:.4f}")
        return score
    print("【4层模型 BLEU 评估】")
    evaluate_bleu(model, test_ds, num_samples=100)


开始训练 (4层, AdamW, lr=0.0005)...


D:\python3.12\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


  0%|          | 0/101400 [00:00<?, ?it/s]